# Export ARCSIX Dashboard to HTML

This notebook demonstrates how to export your ARCSIX phase dashboard as a standalone HTML file that can be easily shared with colleagues. The HTML file includes an interactive threshold slider and requires no Python environment to view - just open it in any web browser.

## Prerequisites

Run this notebook **after** completing the analysis in `analysis_arcsix_single_granule.ipynb` or `presentation_arcsix_single_granule.ipynb`. The HTML export uses the same cached index data, so no additional LUT computation or data download is needed.

## 1. Setup: Locate repository and config

Same setup as the presentation notebook. Set `REPO_OVERRIDE` if your kernel doesn't start in the repository.

In [ ]:
from pathlib import Path
import os
import sys

REPO_OVERRIDE = None  # Example: "~/PACE_spectropolarimetric_phase/PACE-specpol-phase"
if REPO_OVERRIDE:
    REPO = Path(REPO_OVERRIDE).expanduser().resolve()
else:
    REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "src/pace_specpol").is_dir()), None)
if REPO is None or not (REPO / "pyproject.toml").is_file():
    raise FileNotFoundError("Set REPO_OVERRIDE to your local PACE-specpol-phase clone.")

# Add repository to Python path if needed (for development installs)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

CONFIG = Path(os.environ.get("PACE_CONFIG", str(REPO / "config/arcsix.local.toml"))).expanduser().resolve()
if not CONFIG.is_file():
    raise FileNotFoundError(f"Select your existing Arctic paths TOML; not found: {CONFIG}")
print("Repository:", REPO)
print("Local configuration:", CONFIG)

## 2. Install package (if needed)

Run this cell after a fresh environment restart or if you get import errors. The editable install uses this kernel's Python. Restart the **kernel** if prompted, then rerun from the beginning.

In [ ]:
%pip install -e "{REPO}[validation]"

## 3. Import dependencies

Import the HTML export function and other required modules.

In [ ]:
try:
    from pace_specpol.paths import load_paths
    from pace_specpol.workflow import VariantReference
    from pace_specpol.validation.aggregation import IndexConfig, build_index
    from pace_specpol.validation.vis_html import export_html_dashboard
except ImportError as e:
    print(f"Import error: {e}")
    print("\nIf you haven't installed the package yet, run:")
    print(f"  %pip install -e \"{REPO}[validation]\"")
    print("\nThen restart the kernel and rerun from the beginning.")
    raise

# Load paths configuration
paths = load_paths(CONFIG)
print("Work directory:", paths["work_root"])
print("Export directory:", paths["export_root"])

## 4. Configure export settings

Customize your HTML dashboard export. The default settings match the presentation notebook.

In [ ]:
# Output settings
EXPORT_LABEL = "arcsix_june10_html"  # Change to create new export
OUTPUT_DIR = paths["export_root"] / EXPORT_LABEL
OVERWRITE = False  # Set True to replace existing export

# Dashboard settings
THRESHOLDS = [1.10, 1.27, 1.45]  # Pre-render these threshold values
DEFAULT_THRESHOLD = 1.27  # Initial threshold to display
MIN_COUNT = 1  # Minimum samples per cell

# Reference settings (must match your cached analysis)
VARIANT = "oci_2260"  # Cloud effective radius / optical thickness variant
POPULATION = "own"  # 'own' uses granule statistics; 'global' uses all-granule statistics
MODE = "normalized"  # 'raw', 'corrected', or 'normalized'

# Visual settings
DPI = 150  # Resolution (150 is good for web, 240 for high-res)
PREVIEW_WIDTH = 900  # Map preview width in pixels
RATIO_CLIM = (0.5, 2.0)  # Ratio colorbar limits
LI_CLIM = (-0.5, 4.0)  # Liquid index colorbar limits
MAP_EXTENT = (-85, -10, 58, 87)  # Arctic region (west, east, south, north)
GRATICULES = True  # Show graticule lines

# Custom title (optional)
TITLE = "ARCSIX June 10, 2024 Phase Dashboard"

print(f"Will export to: {OUTPUT_DIR}")

## 5. Load or build the index

Load your existing cached index from the analysis notebook. If it doesn't exist, you'll need to run the analysis notebook first.

In [ ]:
# Define granule and scene
GRANULE = "20240610T154205"  # June 10, 2024
SCENE = "arctic_june10_2024"

# Try to locate cached data - check both old and new structures
# New structure: separate reference_cache and companion_cache directories
ref_dir = paths["work_root"] / "reference_cache" / VARIANT / GRANULE
companion_dir = paths["work_root"] / "companion_cache" / GRANULE

# Old structure: combined directory with _v1_pilot1 suffix
old_cache_pattern = f"arcsix_{GRANULE}_*_v1_pilot1"
old_caches = list(paths["work_root"].glob(old_cache_pattern))

if ref_dir.exists() and companion_dir.exists():
    # New structure found
    print(f"Using new cache structure:")
    print(f"  Reference: {ref_dir}")
    print(f"  Companion: {companion_dir}")
elif old_caches:
    # Old structure found
    cache_dir = old_caches[0]
    print(f"Found existing cache: {cache_dir.name}")
    # In old structure: companions/ and references/VARIANT/
    companion_dir = cache_dir / "companions"
    ref_dir = cache_dir / "references" / VARIANT
    
    if not companion_dir.exists() or not ref_dir.exists():
        # Show what's actually there
        print(f"\nCache directory contents:")
        for item in sorted(cache_dir.iterdir()):
            print(f"  {item.name}{'/' if item.is_dir() else ''}")
        if not companion_dir.exists():
            print(f"\n  Missing: companions/")
        if not ref_dir.exists():
            print(f"  Missing: references/{VARIANT}/")
            if (cache_dir / "references").exists():
                print(f"\n  Available reference variants:")
                for var in sorted((cache_dir / "references").iterdir()):
                    print(f"    {var.name}")
        raise FileNotFoundError(
            f"Cache structure incomplete for variant '{VARIANT}'.\n"
            f"Check VARIANT setting in cell 4."
        )
    print(f"  Companion: {companion_dir}")
    print(f"  Reference: {ref_dir}")
else:
    # No cache found - list what's available
    print("Available directories in work_root:")
    for item in sorted(paths["work_root"].iterdir()):
        if item.is_dir():
            print(f"  {item.name}")
    raise FileNotFoundError(
        f"No cache found for granule {GRANULE}.\n"
        f"Expected: arcsix_{GRANULE}_*_v1_pilot1/\n\n"
        f"Run analysis_arcsix_single_granule.ipynb to generate cache."
    )

# Configure index settings
index_config = IndexConfig(
    resolution=0.1,  # 0.1 degree grid
    ratio_step=0.01,
    li_threshold=0.3,
)

# Build index from cached data
print("\nBuilding aggregation index...")
reference = VariantReference(variant=VARIANT, population=POPULATION, mode=MODE)
index = build_index(
    companion_dir=companion_dir,
    reference_dir=ref_dir,
    config=index_config,
    reference=reference,
    data_scope=f"Arctic {SCENE}",
)

print(f"Index built: {index.metadata['totals']['indexed_samples']:,} samples")

## 6. Export HTML dashboard

Generate the interactive HTML dashboard. This will create a standalone file that can be shared via email, Slack, or any file sharing service.

In [ ]:
import time

# Create output filename
output_filename = f"dashboard_{VARIANT}_{MODE}_{POPULATION}.html"
output_path = OUTPUT_DIR / output_filename

# Check if output already exists
if output_path.exists() and not OVERWRITE:
    print(f"Output already exists: {output_path}")
    print("Set OVERWRITE=True or change EXPORT_LABEL to create a new export.")
else:
    # Export HTML dashboard
    print(f"Exporting HTML dashboard with {len(THRESHOLDS)} threshold values...")
    print(f"This may take a minute or two...\n")
    
    start_time = time.time()
    
    result = export_html_dashboard(
        index,
        output_path,
        thresholds=THRESHOLDS,
        default_threshold=DEFAULT_THRESHOLD,
        min_count=MIN_COUNT,
        dpi=DPI,
        preview_width=PREVIEW_WIDTH,
        ratio_clim=RATIO_CLIM,
        li_clim=LI_CLIM,
        map_extent=MAP_EXTENT,
        graticules=GRATICULES,
        title=TITLE,
    )
    
    elapsed = time.time() - start_time
    
    print(f"\n✓ Export complete in {elapsed:.1f} seconds!")
    print(f"\nYou can now:")
    print(f"  1. Open {result} in your browser")
    print(f"  2. Share the file with colleagues")
    print(f"  3. Upload to a web server or cloud storage")

## 7. View the HTML dashboard (optional)

Display the HTML dashboard directly in this notebook using an IFrame.

In [ ]:
from IPython.display import IFrame

if output_path.exists():
    # Display the HTML file in the notebook
    IFrame(src=str(output_path), width="100%", height="1000")
else:
    print("HTML file not found. Run the export cell first.")

## Tips for sharing

### Email or Slack
Simply attach the HTML file. Recipients can download and open it in their browser (Chrome, Firefox, Safari, Edge all work).

### Cloud storage
Upload to:
- Google Drive (right-click → Open with → Google Docs won't work; download and open locally)
- Dropbox (share a direct download link)
- OneDrive, Box, etc.

### Internal web server
Upload to your organization's web server to create a permanent URL. The HTML file is completely self-contained with all images embedded.

### File size
- With 3 thresholds at 150 DPI: ~5-8 MB
- With 5 thresholds at 150 DPI: ~8-12 MB
- With 3 thresholds at 240 DPI: ~12-18 MB

Higher DPI values create larger files but better quality for presentations.

## Advanced: Export multiple configurations

You can export multiple HTML dashboards with different settings to compare different reference modes or variants.

In [ ]:
# Example: Export dashboards for different modes
EXPORT_CONFIGS = [
    {"mode": "normalized", "label": "normalized"},
    {"mode": "corrected", "label": "corrected"},
    {"mode": "raw", "label": "raw"},
]

# Uncomment to run batch export
# for config in EXPORT_CONFIGS:
#     print(f"\nExporting {config['label']} dashboard...")
#     reference = VariantReference(variant=VARIANT, population=POPULATION, mode=config["mode"])
#     index = build_index(companion_dir, ref_dir, index_config, reference, data_scope=f"Arctic {SCENE}")
#     
#     output_file = OUTPUT_DIR / f"dashboard_{VARIANT}_{config['label']}_{POPULATION}.html"
#     export_html_dashboard(index, output_file, thresholds=THRESHOLDS, 
#                          default_threshold=DEFAULT_THRESHOLD, title=f"ARCSIX - {config['label'].title()}")
#     print(f"✓ Saved: {output_file}")